<a href="https://colab.research.google.com/github/prithwis/Centaur/blob/main/MOS_Simulation_v0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![alt text](https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/images/CentaurBanner2.png)<br>


<hr>

[Prithwis Mukerjee](http://www.linkedin.com/in/prithwis)<br>

## Middle East Oil Security Simulation Engine

In [1]:
!wget -O mos.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mos.py
!wget -O mos2.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mos2.py


import mos
import mos2

In [2]:
import importlib


importlib.reload(mos)
importlib.reload(mos2)

from mos import *
from mos2 import *

In [2]:
from mos import *
from mos2 import *

In [3]:
client = authenticateOpenAI()
model = "gpt-5.6-luna"

OpenAI authentication successful ✔
Logged in as Calcutta prithwis@yantrajaal.com


In [4]:
worldVariables = [

    WorldVariable(
        name="Oil Price",
        description=(
            "International crude oil price level, represented "
            "by the Brent benchmark."
        ),
        direction="Higher index means higher oil price."
    ),

    WorldVariable(
        name="Hormuz Oil Flow",
        description=(
            "Volume of crude oil and petroleum products successfully "
            "transiting the Strait of Hormuz."
        ),
        direction="Higher index means greater oil flow."
    ),

    WorldVariable(
        name="India Gulf Oil Imports",
        description=(
            "Volume of crude oil imported by India from "
            "Gulf-origin oil-producing countries."
        ),
        direction=(
            "Higher index means greater Indian imports "
            "of Gulf-origin oil."
        )
    ),

    WorldVariable(
        name="Iran Sanctions Enforcement",
        description=(
            "Degree to which restrictions on Iranian petroleum exports, "
            "trade and financial transactions are effectively enforced "
            "rather than circumvented."
        ),
        direction=(
            "Higher index means more effective sanctions enforcement "
            "and less successful circumvention."
        )
    ),

    WorldVariable(
        name="US/Allied Cohesion",
        description=(
            "Degree of alignment between the United States and cooperating "
            "states in their response to Iran and the Gulf crisis."
        ),
        direction="Higher index means greater US/allied cohesion."
    ),

    WorldVariable(
        name="Iran Internal Cohesion",
        description=(
            "Degree of alignment within Iran's political, military and "
            "security establishment in responding to the crisis."
        ),
        direction="Higher index means greater internal Iranian cohesion."
    ),

    WorldVariable(
        name="US-Iran Tension",
        description=(
            "Intensity of strategic hostility between the United States "
            "and Iran across diplomatic, economic and military domains."
        ),
        direction="Higher index means greater US-Iran tension."
    ),

    WorldVariable(
        name="Regional Armed Conflict",
        description=(
            "Intensity and geographic breadth of kinetic military activity "
            "associated with the regional crisis."
        ),
        direction=(
            "Higher index means more intense or geographically "
            "widespread armed conflict."
        )
    )
]

In [5]:
from google.colab import drive
drive.mount('/content/drive')

WORLDFACTS_DIR = "/content/drive/MyDrive/Praxis/Course Geopolitics/Centaur Project/WorldFacts"

!ls -l "{WORLDFACTS_DIR}"
!ls -lh "{WORLDFACTS_DIR}/MD"
!ls -lh "{WORLDFACTS_DIR}/NEWS"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
total 12
drwx------ 2 root root 4096 Sep 28 09:09 MD
drwx------ 2 root root 4096 Sep 29 09:42 NEWS
drwx------ 2 root root 4096 Sep 28 06:55 PDF
total 230K
-rw------- 1 root root 7.6K Sep 29 13:39 IranData_260929.md
-rw------- 1 root root 6.5K Sep 29 13:39 MissilesofIranMissil_260929.md
-rw------- 1 root root 8.2K Sep 29 13:40 OilMarketReportSepte_260929.md
-rw------- 1 root root  12K Sep 29 13:40 PressReleasePagePres_260929.md
-rw------- 1 root root 4.2K Sep 29 13:43 ShortTermEnergyOutlo_260929.md
-rw------- 1 root root  76K Sep 29 13:43 SIPRISummary_260929.md
-rw------- 1 root root  40K Sep 29 13:45 SnapshotofIndiasOilG_260929.md
-rw------- 1 root root  18K Sep 29 13:45 UKCommonsIranin2026S_260929.md
-rw------- 1 root root  57K Sep 29 13:47 USCongressR488873_260929.md
total 50K
-rw------- 1 root root 50K Sep 29 13:54 MOS-MC-LiveFeed_260929_1924_IST.txt


In [6]:
WorldFactBase = getWorldFactBase(WORLDFACTS_DIR)


Static sources : 9
Live sources   : 1
Characters     : 281,993
Words          : 43,548


In [7]:
ZW_Role = """
You are ZeitWorld, the neutral state-transition engine in a geopolitical
simulation of Middle East oil security.

Your responsibility is to describe the state of the simulated world.
You do not recommend policy, choose strategies, judge whether an actor
has acted wisely, or predict what an actor should do next.

A WorldState consists of a set of WorldVariables. Each WorldVariable has:

- name: the identity of the variable.
- description: the precise meaning of the variable.
- direction: the meaning of an increase in its index.
- index: a normalized numerical representation of its current state.
- status: a concise description of its current condition.
- conditions: a list of previously created conditions that remain
  causally operative in the current world.

The fields name, description and direction define the variable and must
never be changed during a simulation run.

The fields index, status and conditions describe the current state and
may change as the simulated world changes.

INDEX INTERPRETATION

Z0 is the baseline WorldState. Every variable has index = 100 at Z0.

All subsequent indices remain fixed-base indices relative to Z0.
They are never rebased against the immediately preceding state.

For quantitatively measurable variables, an index of 120 should mean
approximately 20 percent above the Z0 baseline and an index of 80 should
mean approximately 20 percent below it.

For qualitative variables, the index represents a disciplined relative
assessment of intensity compared with Z0. Do not imply false numerical
precision.

STATE MEMORY

ZeitWorld has no access to strategic history.

The current WorldState is therefore authoritative and must contain all
conditions from earlier events that remain causally relevant.

The conditions field is NOT an event log or narrative history.
Retain a condition only while its consequences remain operative.
Modify it as the condition evolves and remove it when it no longer
affects the current world.

NEUTRALITY

Be a cold, neutral state recorder.

Do not favour the United States, Iran, India, or any other actor.
Do not infer that an action was intelligent, foolish, justified,
successful or unsuccessful except insofar as its observable consequences
must be represented in the resulting WorldState.

Distinguish between an action and its consequences.
An executed action is a fact of the simulated world. Your task is to
determine what state follows from it.

Do not invent dramatic consequences merely to make the simulation
interesting. Preserve unchanged variables when there is insufficient
causal reason for them to move.

OUTPUT DISCIPLINE

When asked to produce a WorldState, return only valid JSON conforming
exactly to the supplied WorldState structure.

Do not add variables.
Do not remove variables.
Do not rename variables.
Do not add explanatory prose outside the JSON.
"""

In [8]:
ZeitWorld = ZeitWorld_Agent(client, model, ZW_Role)

In [9]:
print(ZeitWorld.initialised)
print(ZeitWorld.model)

False
gpt-5.6-luna


In [10]:
W = []

W.append(
    WorldState(
        state_id="W0",
        variables=worldVariables
    )
)

print(W[0].to_json())

{
  "state_id": "W0",
  "variables": [
    {
      "name": "Oil Price",
      "description": "International crude oil price level, represented by the Brent benchmark.",
      "direction": "Higher index means higher oil price.",
      "index": 100,
      "status": "",
      "conditions": []
    },
    {
      "name": "Hormuz Oil Flow",
      "description": "Volume of crude oil and petroleum products successfully transiting the Strait of Hormuz.",
      "direction": "Higher index means greater oil flow.",
      "index": 100,
      "status": "",
      "conditions": []
    },
    {
      "name": "India Gulf Oil Imports",
      "description": "Volume of crude oil imported by India from Gulf-origin oil-producing countries.",
      "direction": "Higher index means greater Indian imports of Gulf-origin oil.",
      "index": 100,
      "status": "",
      "conditions": []
    },
    {
      "name": "Iran Sanctions Enforcement",
      "description": "Degree to which restrictions on Iranian petro

In [11]:
#Z0 = WorldState(state_id="Z0",variables=worldVariables)
#print(Z0.to_json())

In [12]:
ZW_Initialisation_Input = f"""
MODE: INITIALISE Z0

You are creating the initial WorldState Z0.

Z0 represents the real-world condition at the exact point at which the
simulation begins.

You are given:

1. A WorldState template containing the WorldVariables that define this
   simulation.
2. WORLD FACTS collected from selected static sources and a current
   news feed.

TASK

For each WorldVariable:

- Preserve name exactly.
- Preserve description exactly.
- Preserve direction exactly.
- Keep index exactly 100. Z0 is the normalized baseline.
- Populate status with a concise assessment of the current real-world
  condition of that variable.
- Populate conditions with only important currently operative conditions
  that may continue to affect future state transitions.

STATUS

Status should normally contain no more than three concise sentences.

It should describe what the supplied evidence establishes about the
variable at the simulation starting point.

Do not turn status into a history, explanation of the variable, policy
assessment, or prediction.

CONDITIONS

Conditions are pieces of state memory that future ZeitWorld transitions
may need even though they will not have access to these WORLD FACTS.

Include a condition only when:

- it is supported by the supplied WORLD FACTS;
- it is currently operative; and
- its continuing existence could materially affect a future WorldState.

Do not use conditions as a summary of the sources.
Do not duplicate the status unnecessarily.
An empty conditions list is valid.

EVIDENCE DISCIPLINE

Use only the supplied WORLD FACTS.

Do not use outside knowledge.
Do not assume that the premise implicit in a variable is true.
Look for supporting and contradictory evidence where relevant.
If evidence for a variable is weak or conflicting, state that uncertainty
in status rather than inventing certainty.

The LIVE SOURCE represents the most recent evidence and may update or
supersede older static-source information when they genuinely conflict.
Do not automatically prefer the LIVE SOURCE when the difference is not
time-dependent.

Return only the completed WorldState as valid JSON.

WORLDSTATE TEMPLATE
===================

{W[0].to_json()}


WORLD FACTS
===========

{WorldFactBase}
"""

In [13]:
W[0] = ZeitWorld.initialise(W[0], WorldFactBase, ZW_Initialisation_Input)

Model: gpt-5.6-luna | Effort: medium | Input: 131,285 | Output: 1,887 | Reasoning: 128 | Total: 133,172


In [14]:
print(W[0].to_json())

{
  "state_id": "W0",
  "variables": [
    {
      "name": "Oil Price",
      "description": "International crude oil price level, represented by the Brent benchmark.",
      "direction": "Higher index means higher oil price.",
      "index": 100,
      "status": "Brent crude averaged approximately $91 per barrel in August 2026 and reached about $113.48 per barrel on 9 September. Prices remain substantially above pre-conflict levels amid constrained Middle Eastern exports, attacks on shipping, and depleted inventories.",
      "conditions": [
        "Middle Eastern production and export disruptions remain operative, with more than 10 million barrels per day of Gulf output reported shut in during August.",
        "Global observed oil inventories have fallen by approximately 507 million barrels since February, while tanker costs and security risks remain elevated.",
        "The United States-Iran confrontation and associated disruptions at the Strait of Hormuz and Bab el-Mandeb contin

In [16]:
print(ZeitWorld.initialised)
print(ZeitWorld.model)

True
gpt-5.6-luna
